<a href="https://colab.research.google.com/github/raisa1521/Hands-on-Machine-Learning/blob/main/8%EC%9E%A5_%EC%B0%A8%EC%9B%90%EC%B6%95%EC%86%8C%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**8장 차원 축소**

- 특성이 너무 많으면 훈련이 느려지고 좋은 솔루션을 찾기 어려움
- **차원 축소**: 특성 수를 줄여 훈련 속도 높임. 2차원이나 3차원으로 줄이면 군집 같은 패턴을 시각화 가능
- MNIST의 경계 픽셀은 거의 항상 흰색이고 인접 픽셀은 강하게 연관 -> 제거하거나 합쳐도 잃는 정보 적음
- 일부 정보가 유실되어 성능이 나빠질 수 있어서 먼저 원본 데이터로 훈련 권장. 잡음이나 불필요한 세부 사항을 걸러내 성능이 높아지는 경우도 있지만 일반적으로는 훈련 속도만 빨라짐.

#**8.1 차원의 저주**

- 고차원 공간에서는 대부분의 점이 경계와 가까이 있고, 샘플 사이의 거리가 커짐
- 데이터셋이 매우 희박해져 훈련 샘플들이 서로 멀리 떨어짐.
- 새로운 샘플도 훈련 샘플과 멀리 떨어질 가능성이 높음 -> 예측에 더 많은 외삽이 필요하고 예측이 불안정해짐

-> 훈련 세트의 차원이 클수록 과대적합 위험이 커짐.

훈련 샘플의 밀도를 높이려면 샘플 수를 늘려야 하지만 필요한 샘플 수가 차원 수에 따라 기하급수적으로 증가

#**8.2 차원 축소를 위한 접근법**

## **8.2.1 투영**

- 실전에서는 변화가 거의 없는 특성이나 서로 강하게 연관된 특성이 많음.
- 훈련 샘플이 고차원 공간 안의 저차원 부분공간 또는 그 가까이에 놓임 -> 이 부분공간에 투영하여 차원을 줄임.
- 스위스 롤처럼 부분공간이 휘거나 뒤틀려 있으면 평면에 투영할 때 층이 서로 뭉개짐. 이 경우엔 매니폴드를 펼쳐야 함.

### **8.2.2 매니폴드 학습**

- **매니폴드**: 국부적으로 d차원 초평면으로 보이는 n차원 공간의 일부(d < n) 스위스 롤은 3차원 공간에 말려 있는 2차원 매니폴드.
- **매니폴드 학습**: 훈련 샘플이 놓여 있는 매니폴드를 모델링
- **매니폴드 가정 또는 매니폴드 가설**: 실제 고차원 데이터셋은 더 낮은 차원의 매니폴드에 가깝게 놓임

-> 저차원 매니폴드에서 결정 경계가 더 단순해질 수도 있지만 더 복잡해질 수도 있음. 차원 축소가 항상 더 낫거나 간단한 솔루션을 만드는 것은 아님. 데이터셋에 달림.

# **8.3 주성분 분석**

**주성분 분석(PCA)**: 데이터에 가장 가까운 초평면을 정의하고 데이터를 이 평면에 투영함.

### **8.3.1 분산 보존**

PCA에서는 데이터를 투영할 초평면을 선택

-> **분산을 가장 많이 보존하는 축** 선택

-> 분산을 최대한 보존하면 정보 손실을 최소화

다르게 표현하면 원본 데이터셋과 투영된 데이터셋 사이의 평균 제곱 거리를 최소화하는 축을 선택하는 것

### **8.3.2 주성분**

PCA는 훈련 세트에서 분산이 가장 큰 축을 찾고, 첫 번째 축에 직교하면서 남은 분산을 최대한 보존하는 두 번째 축을 찾음.

고차원 데이터에서는 같은 방식으로 계속 새로운 축을 찾음.

i번째 축을 데이터의 i번째 주성분(PC)라고 함

각 주성분에 대해 PCA는 주성분 방향을 나타내는 단위 벡터를 찾음

훈련 세트 행렬 X는 특잇값 분해(SVD)를 이용해 다음과 같이 분해할 수 있음.

$\mathbf{X}=\mathbf{U}\mathbf{\Sigma}\mathbf{V}^T$

모든 주성분의 단위 벡터가 $\mathbf{V}$에 포함됨.

$\mathbf{V}=[\mathbf{c}_1\ \mathbf{c}_2\ \cdots\ \mathbf{c}_n]$

In [7]:
import numpy as np

# 작은 3D 데이터셋을 만듭니다 (예시 데이터)
np.random.seed(42)
m = 60
w1, w2 = 0.1, 0.3
noise = 0.1

angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)

X_centered = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(X_centered)
c1 = Vt[0]
c2 = Vt[1]

PCA는 데이터셋의 평균이 0이라고 가정

-> 사이킷런의 `PCA`는 데이터를 자동으로 중앙에 맞춰줌

-> 직접 PCA를 구현하거나 다른 라이브러리를 사용한다면 먼저 데이터를 원점에 맞춰야 함

### **8.3.3 d차원으로 투영하기**

처음 d개의 주성분으로 정의된 초평면에 데이터를 투영하면 데이터셋을 d차원으로 축소할 수 있음.

**- 훈련 세트를 d차원으로 투영하기 -**

$\mathbf{X}_{d\text{-proj}}=\mathbf{X}\mathbf{W}_d$

In [8]:
W2 = Vt[:2].T
X2D = X_centered @ W2

### **8.3.4 사이킷런 사용하기**

사이킷런의 PCA는 SVD를 사용하며 데이터를 자동으로 중앙에 맞춰줌.

In [10]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

#PCA를 학습한 뒤 components_에는 $\mathbf{W}_d$의 전치가 저장됨-> 각 행이 처음 d개의 주성분에 해당

### **8.3.5 설명된 분산의 비율**

explained_variance_ratio_ : 각 주성분 축을 따라 존재하는 데이터셋의 분산 비율

In [11]:
pca.explained_variance_ratio_

array([0.85406025, 0.13622918])

->데이터셋 분산의 약 76%가 첫 번째 PC를 따라 놓여 있고 약 15%가 두 번째 PC를 따라 놓여 있음.

### **8.3.6 적절한 차원 수 선택**

축소할 차원 수를 임의로 정하기보다 충분한 분산을 보존할 수 있는 차원 수를 선택하는 것이 좋음

ex) 분산의 95%를 보존하는 데 필요한 최소 차원 수를 계산

In [12]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False, parser='auto')
X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]

pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1 # d == 154

유지하려는 주성분 수를 직접 지정하기보다 `n_components`에 보존하려는 분산 비율을 설정할 수 있음

In [14]:
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)

In [15]:
pca.n_components_
#결과 - 154

np.int64(154)

또 다른 방법 = 설명된 분산을 차원 수에 대한 함수로 그린 뒤 변곡점(엘보)을 찾는 것

지도 학습의 전처리 단계로 PCA를 사용하는 경우 차원 수도 다른 하이퍼파라미터처럼 튜닝할 수 있음

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

clf = make_pipeline(PCA(random_state=42),
                    RandomForestClassifier(random_state=42))

param_distrib = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

rnd_search = RandomizedSearchCV(clf, param_distrib, n_iter=10, cv=3,
                                random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('pca', PCA(random_state=42)),
                                             ('randomforestclassifier',
                                              RandomForestClassifier(random_state=42))]),
                   param_distributions={'pca__n_components': array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26,
       27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43,
       44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60,
       6...
       414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426,
       427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439,
       440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452,
       453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465,
       466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478,
       479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491,
       492, 493, 494, 495, 496, 497, 498, 499])},
                   random_state=42)

In [17]:
print(rnd_search.best_params_)
#결과 - {'randomforestclassifier__n_estimators': 465, 'pca__n_components': 23}

{'randomforestclassifier__n_estimators': np.int64(475), 'pca__n_components': np.int64(57)}


-> 이 예에서는 784차원의 데이터를 23차원으로 줄이는 조합이 선택

### **8.3.7 압축을 위한 PCA**

PCA로 차원을 축소하면 데이터셋이 차지하는 공간을 크게 줄일 수 있음

MNIST에서 분산의 95%를 유지하면

784개 특성 -> 154개 특성

-> 원본 크기의 20% 이하로 줄이면서 분산은 5%만 손실

축소된 데이터에 PCA 투영의 역변환을 적용해 원래 차원 수로 되돌릴 수 있음.

하지만 투영 과정에서 정보가 손실되었으므로 원본과 완전히 같지는 않음.

**재구성 오차**: 원본 데이터와 압축 후 복원된 데이터 사이의 평균 제곱 거리

In [18]:
X_recovered = pca.inverse_transform(X_reduced)

**원본의 차원 수로 되돌리는 PCA 역변환**

$\mathbf{X}_{\text{recovered}}=\mathbf{X}_{d\text{-proj}}\mathbf{W}_d^T$

### **8.3.8 랜덤 PCA**

`svd_solver="randomized"`로 지정하면 **랜덤 PCA**를 사용하여 처음 d개의 주성분의 근삿값을 빠르게 찾음.

완전한 SVD의 계산 복잡도

$O(mn^2)+O(n^3)$

랜덤 PCA의 계산 복잡도

$O(md^2)+O(d^3)$

-> d가 n보다 많이 작으면 완전 SVD보다 훨씬 빠름

In [19]:
rnd_pca = PCA(n_components=154, svd_solver="randomized",
              random_state=42)
X_reduced = rnd_pca.fit_transform(X_train)

`svd_solver`의 기본값은 `"auto"`

`max(m,n)>500`이고 `n_components`가 $\min(m,n)$의 80%보다 작은 정수이면 랜덤 PCA를 자동으로 사용

완전한 SVD를 사용하려면 `svd_solver="full"`로 설정

### **8.3.9 점진적 PCA**

기본 PCA는 SVD를 수행하기 위해 전체 훈련 세트를 메모리에 올려야 함

**점진적 PCA(IPCA)**

- 훈련 세트를 미니배치로 나누어 한 번에 하나씩 주입
- 훈련 세트가 클 때 유용
- 새로운 데이터가 준비되는 대로 온라인으로 PCA를 적용할 수도 있음

In [20]:
from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X_train)
#디스크에 저장된 대규모 배열을 메모리에 있는 것처럼 사용할 수 있는 넘파이 memmap 이용할 수도 있음.

In [21]:
filename = "my_mnist.mmap"
X_mmap = np.memmap(filename, dtype='float32', mode='write',
                   shape=X_train.shape)
X_mmap[:] = X_train # 대신 반복을 사용해 데이터를 한 청크씩 저장할 수 있습니다.
X_mmap.flush()

In [22]:
X_mmap = np.memmap(filename, dtype="float32",
                   mode="readonly").reshape(-1, 784)
batch_size = X_mmap.shape[0] // n_batches
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mmap)

IncrementalPCA(batch_size=600, n_components=154)

# **8.4 랜덤 투영**

**랜덤 투영**: 랜덤한 선형 투영을 사용해 데이터를 저차원 공간에 투영하는 방법

랜덤 투영은 거리를 상당히 잘 보존할 가능성이 높음

-> 비슷한 샘플은 투영 후에도 비슷하게 남고, 서로 다른 샘플은 서로 다르게 남음

더 많은 차원을 제거할수록 정보 손실과 거리 왜곡은 증가

거리가 주어진 허용 오차 $\epsilon$ 이상 변하지 않도록 보존할 최소 차원 수는 다음과 같이 정할 수 있음.

$d\geq\frac{4\log(m)}{\frac{1}{2}\epsilon^2-\frac{1}{3}\epsilon^3}$

이 식은 원본 특성 수 n이 아니라 샘플 수 $m$과 허용 오차 $\epsilon$에 의존함.

In [24]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim

m, ε = 5_000, 0.1
d = johnson_lindenstrauss_min_dim(m, eps=ε)
d
#결과 - 7300
#가우스 분포에서 랜덤하게 생성한 [d,n]크기의 랜덤 행렬을 사용해 n차원 데이터를 d차원으로 투영할 수 있음.

np.int64(7300)

In [25]:
n = 20_000
np.random.seed(42)
P = np.random.randn(d, n) / np.sqrt(d) # 표준 편차 = 분산의 제곱근

X = np.random.randn(m, n)              # 가짜 데이터셋을 생성합니다.
X_reduced = X @ P.T

랜덤 행렬 생성에는 데이터셋의 크기만 필요함.

-> 훈련이 필요하지 않고 데이터 자체도 사용하지 않음.

In [26]:
from sklearn.random_projection import GaussianRandomProjection

gaussian_rnd_proj = GaussianRandomProjection(eps=ε, random_state=42)
X_reduced = gaussian_rnd_proj.fit_transform(X) # 이전 결과와 동일합니다.

**SparseRandomProjection**

- 랜덤 행렬이 희소
- 메모리를 훨씬 적게 사용
- 랜덤 행렬 생성과 차원 축소가 더 빠름
- 입력이 희소하면 `dense_output=True`로 설정하지 않는 한 희소성을 유지
- 거리 보존 성질과 차원 축소 품질은 비슷

-> 특히 규모가 크거나 희박한 데이터셋에서 유용

희소 랜덤 행렬에서 0이 아닌 항목의 비율 = **밀도(density)** r

기본값

$r=\frac{1}{\sqrt{n}}$

0이 아닌 값은 -v 또는 +v이며

$v=\frac{1}{\sqrt{dr}}$

랜덤 투영의 역변환은 성분 행렬의 유사역행렬을 구한 뒤 축소된 데이터에 유사역행렬의 전치를 곱함.

In [ ]:
components_pinv = np.linalg.pinv(gaussian_rnd_proj.components_)
X_recovered = X_reduced @ components_pinv.T

-> 랜덤 투영은 간단하고 빠르며 메모리 효율이 높은 차원 축소 방법으로 특히 고차원 데이터셋에서 유용

# **8.5 지역 선형 임베딩**

**지역 선형 임베딩(LLE)**: 투영에 의존하지 않는 **비선형 차원 축소** 매니폴드 학습 방법

1. 각 훈련 샘플이 최근접 이웃에 얼마나 선형적으로 연관되어 있는지 측정
2. 이 국부적 관계가 가장 잘 보존되는 저차원 표현을 찾음

-> 특히 잡음이 너무 많지 않은 경우 꼬인 매니폴드를 펼치는 데 적합

In [ ]:
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding

X_swiss, t = make_swiss_roll(n_samples=1000, noise=0.2,
                             random_state=42)
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10,
                             random_state=42)
X_unrolled = lle.fit_transform(X_swiss)

결과적으로 스위스 롤이 펼쳐지고 지역적으로는 샘플 간 거리가 잘 보존

하지만 전체적으로 샘플 간 거리가 완전히 보존되는 것은 아님.

**LLE 단계 1: 선형적인 지역 관계 모델링**

각 훈련 샘플 $\mathbf{x}^{(i)}$에 대해 k개의 최근접 이웃을 찾고, 이 이웃들의 선형 결합으로 $\mathbf{x}^{(i)}$를 재구성하는 가중치 $w_{i,j}$를 찾음.

$\hat{\mathbf{W}}
=
\underset{\mathbf{W}}{\operatorname{argmin}}
\sum_{i=1}^{m}
\left(
\mathbf{x}^{(i)}
-
\sum_{j=1}^{m}w_{i,j}\mathbf{x}^{(j)}
\right)^2$

조건

$w_{i,j}=0$

$\mathbf{x}^{(j)}$가 $\mathbf{x}^{(i)}$의 최근접 이웃 k개 중 하나가 아닐 때

그리고

$\sum_{j=1}^{m}w_{i,j}=1
\qquad i=1,2,\ldots,m$

-> 가중치 행렬 $\mathbf{W}$가 훈련 샘플 사이의 지역 선형 관계를 나타냄.

**LLE 단계 2: 관계를 보존하는 차원 축소**

첫 번째 단계에서 구한 가중치를 고정한 뒤 저차원 공간에서 이 관계를 가장 잘 보존하는 샘플 위치 $\mathbf{z}^{(i)}$를 찾음

$\hat{\mathbf{Z}}
=
\underset{\mathbf{Z}}{\operatorname{argmin}}
\sum_{i=1}^{m}
\left(
\mathbf{z}^{(i)}
-
\sum_{j=1}^{m}\hat{w}_{i,j}\mathbf{z}^{(j)}
\right)^2$

-> LLE는 투영 기법보다 복잡하지만 비선형 데이터에서 더 나은 저차원 표현을 만들 수 있음

# **8.6 다른 차원 축소 기법**

**다차원 스케일링(MDS)**

- 샘플 간의 거리를 보존하면서 차원을 축소
- 랜덤 투영은 고차원 데이터에 적합하지만 저차원 데이터에서는 잘 작동하지 않음


**Isomap**

- 각 샘플을 가장 가까운 이웃과 연결해 그래프를 만듦.
- 샘플 사이의 지오데식 거리를 유지하면서 차원을 축소
- 그래프에서 두 노드 사이의 지오데식 거리는 두 노드를 잇는 최단 경로의 거리


**t-SNE**

- 비슷한 샘플은 가깝게, 비슷하지 않은 샘플은 멀리 떨어지도록 차원을 축소
- 주로 시각화에 사용
- 특히 고차원 공간의 샘플 군집을 시각화할 때 사용


**선형 판별 분석(LDA)**

- 선형 분류 알고리즘
- 훈련 과정에서 클래스 사이를 가장 잘 구분하는 축을 학습
- 이 축을 데이터가 투영될 초평면을 정의하는 데 사용할 수 있음.
- 투영 후 클래스가 가능한 한 멀리 떨어지도록 유지하므로 다른 분류 알고리즘을 적용하기 전 차원 축소에 사용할 수 있음.